In [ ]:
import time
import pandas as pd
from pathlib import Path
from torch.utils.tensorboard import SummaryWriter
import json

In [ ]:
LOG = Path("runs/rf_detr_v2/log.txt")
writer = SummaryWriter("runs/rf_detr_v2/tensorboard")

seen_epochs = set()

while True:
    if LOG.exists():
        with open(LOG) as f:
            for line in f:
                d = json.loads(line)
                epoch = d["epoch"]

                if epoch in seen_epochs:
                    continue

                seen_epochs.add(epoch)

                coco = d["test_coco_eval_bbox"]

                writer.add_scalar("loss/train", d["train_loss"], epoch)
                writer.add_scalar("loss/val", d["test_loss"], epoch)

                writer.add_scalar("mAP", coco[0], epoch)
                writer.add_scalar("mAP50", coco[1], epoch)

                writer.add_scalar("mAP/small", coco[3], epoch)
                writer.add_scalar("mAP/medium", coco[4], epoch)
                writer.add_scalar("mAP/large", coco[5], epoch)

                writer.flush()

    time.sleep(5)

In [ ]:
LOG = Path("lightning_logs/version_0/metrics.csv")

writer = SummaryWriter("runs/deepforest_v2_neg/tensorboard")

seen = set()

results = []

while True:

    if LOG.exists():

        df = pd.read_csv(LOG)

        df = df[["epoch","train_loss_epoch","val_loss"]]

        for _, row in df.iterrows():

            epoch = row["epoch"]

            if pd.isna(epoch):
                continue

            epoch = int(epoch)

            if epoch in seen:
                continue

            train_loss = row["train_loss_epoch"]
            val_loss = row["val_loss"]

            if pd.isna(train_loss) and pd.isna(val_loss):
                continue

            seen.add(epoch)

            if not pd.isna(train_loss):
                writer.add_scalar("Loss/train", train_loss, epoch)

            if not pd.isna(val_loss):
                writer.add_scalar("Loss/val", val_loss, epoch)

            results.append({
                "epoch": epoch,
                "train_loss": train_loss,
                "val_loss": val_loss
            })

            writer.flush()

            print(epoch, train_loss, val_loss)

        pd.DataFrame(results).to_csv(
            "runs/deepforest_v2_neg/results.csv",
            index=False
        )

    time.sleep(5)